# Experimento 04: Validação de Sensibilidade ao Ruído (0,5) — Pan para Pan (Intra-Dataset)

## Objetivo
Avaliar se a inserção do ruído sentinela neutro (`0,5`) nos genes do **Pan** que estão ausentes em **Fujita** ou **Mathys** degrada a capacidade preditiva do classificador celular **MLP** treinado sobre as projeções limpas do SWeeP 600D.

- **Dataset de Treino (Referência Limpa):** Pan (`57.457` células válidas, classes 1–7)
- **Dataset de Teste com Ruído:** Pan com `0,5` nas colunas ausentes na união de ausências (`G_Pan \ (G_Pan ∩ G_Fujita ∩ G_Mathys)`)
- **Espaço de Projeção:** `61.541` genes $\to$ 600D com `orthbase_pan_600d.npy`
- **Validação:** Teste Holdout (15% sem vazamento) e Dataset Completo (100%)


In [1]:
import gc
import json
import os
import time
from pathlib import Path

import anndata as ad
import cupy as cp
import joblib
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import scipy.sparse as sp
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import sweep
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.models import Sequential, load_model

# Garantir que o diretório de execução seja a raiz do projeto
if Path.cwd().name == "experimentos":
    os.chdir(Path.cwd().parent)
ROOT_DIR = Path.cwd()
DADOS_DIR = ROOT_DIR / "dados"

SEED = 42
DIM_PROJECAO = 600
CLASSES_VALIDAS = [1, 2, 3, 4, 5, 6, 7]
NOMES_CLASSES = [f"Classe {i}" for i in CLASSES_VALIDAS]

np.random.seed(SEED)
tf.random.set_seed(SEED)
tf.config.threading.set_intra_op_parallelism_threads(2)
tf.config.threading.set_inter_op_parallelism_threads(1)

assert sweep.is_gpu_available(), "ERRO: GPU CUDA não detectada pelo pacote sweep!"
print(f"Diretório raiz ativo: {ROOT_DIR}")
print(f"Dispositivo GPU     : {sweep.get_gpu_device_name()}")
print(f"Versão SWeeP        : {getattr(sweep, 'DEFAULT_VERSION', 'v2.2.0')}")


2026-10-24 13:26:18.875676: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Diretório raiz ativo: /home/rasert/workspace/Teste_MlP_fujita_para_fujitafaltante_fujitapos_imputacao
Dispositivo GPU     : NVIDIA GeForce RTX 3060
Versão SWeeP        : SWeeP v2.9


In [2]:
def carregar_ensembl_ids(path_tsv: Path) -> list[str]:
    df = pd.read_csv(path_tsv, sep="\t", header=None)
    return df[0].astype(str).str.strip().str.replace(r"\.\d+$", "", regex=True).tolist()

def carregar_rotulos_filtrados(path_labels: Path) -> tuple[np.ndarray, np.ndarray]:
    with open(path_labels, "r", encoding="utf-8") as f:
        rotulos = np.array([int(float(l.strip().split(",")[0])) for l in f if l.strip()], dtype=np.int64)
    mascara = np.isin(rotulos, CLASSES_VALIDAS)
    y_filtrado = rotulos[mascara] - 1  # 1..7 -> 0..6
    return mascara, y_filtrado

def construir_modelo_mlp(input_dim: int = 600, n_classes: int = 7) -> Sequential:
    model = Sequential([
        Input(shape=(input_dim,)),
        Dense(128, activation="relu", name="camada_oculta_1"),
        Dense(64, activation="relu", name="camada_oculta_2"),
        Dense(32, activation="relu", name="camada_oculta_3"),
        Dense(n_classes, activation="softmax", name="camada_saida"),
    ], name="MLP_Classificador_Celular_600D")
    model.compile(
        optimizer="adam",
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

def plotar_curvas_aprendizado(history, titulo: str):
    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Acurácia ao Longo das Épocas", "Perda (Loss) ao Longo das Épocas")
    )
    epocas = list(range(1, len(history.history["accuracy"]) + 1))
    fig.add_trace(go.Scatter(x=epocas, y=history.history["accuracy"], mode="lines+markers", name="Treino Acurácia", line=dict(color="#1f77b4")), row=1, col=1)
    fig.add_trace(go.Scatter(x=epocas, y=history.history["val_accuracy"], mode="lines+markers", name="Validação Acurácia", line=dict(color="#ff7f0e")), row=1, col=1)
    fig.add_trace(go.Scatter(x=epocas, y=history.history["loss"], mode="lines+markers", name="Treino Loss", line=dict(color="#1f77b4")), row=1, col=2)
    fig.add_trace(go.Scatter(x=epocas, y=history.history["val_loss"], mode="lines+markers", name="Validação Loss", line=dict(color="#ff7f0e")), row=1, col=2)
    fig.update_layout(title_text=titulo, template="plotly_white", width=1000, height=450, legend=dict(orientation="h", y=-0.15))
    fig.update_xaxes(title_text="Época", dtick=1)
    fig.update_yaxes(title_text="Acurácia", row=1, col=1)
    fig.update_yaxes(title_text="Perda", row=1, col=2)
    return fig

def plotar_matriz_confusao(y_true: np.ndarray, y_pred: np.ndarray, titulo: str):
    cm_norm = confusion_matrix(y_true, y_pred, normalize="true")
    cm_abs = confusion_matrix(y_true, y_pred)
    text_labels = [[f"{abs_v}<br>({norm_v*100:.1f}%)" for abs_v, norm_v in zip(r_abs, r_norm)] for r_abs, r_norm in zip(cm_abs, cm_norm)]
    fig = go.Figure(data=go.Heatmap(
        z=cm_norm,
        x=NOMES_CLASSES,
        y=NOMES_CLASSES,
        colorscale="Blues",
        text=text_labels,
        texttemplate="%{text}",
        colorbar=dict(title="Recall"),
        zmin=0, zmax=1
    ))
    fig.update_layout(
        title=titulo,
        xaxis_title="Classe Predita",
        yaxis_title="Classe Real",
        yaxis_autorange="reversed",
        template="plotly_white",
        width=650, height=550
    )
    return fig

def plotar_comparativo(rel_limpo: dict, rel_ruido: dict, titulo: str, nome_cenario_ruido: str = "Com Ruído 0,5"):
    rotulos_plot = ["Acurácia Global", "Macro F1", "Weighted F1"] + [f"F1 {nome}" for nome in NOMES_CLASSES]
    
    vals_limpo = [
        rel_limpo["accuracy"],
        rel_limpo["macro avg"]["f1-score"],
        rel_limpo["weighted avg"]["f1-score"]
    ] + [rel_limpo.get(nome, {}).get("f1-score", 0.0) for nome in NOMES_CLASSES]
    
    vals_ruido = [
        rel_ruido["accuracy"],
        rel_ruido["macro avg"]["f1-score"],
        rel_ruido["weighted avg"]["f1-score"]
    ] + [rel_ruido.get(nome, {}).get("f1-score", 0.0) for nome in NOMES_CLASSES]
    
    fig = go.Figure(data=[
        go.Bar(name="Baseline (Sem Ruído)", x=rotulos_plot, y=vals_limpo, marker_color="#2ca02c"),
        go.Bar(name=nome_cenario_ruido, x=rotulos_plot, y=vals_ruido, marker_color="#d62728")
    ])
    fig.update_layout(
        title=titulo,
        barmode="group",
        yaxis=dict(title="Pontuação (0.0 a 1.0)", range=[0, 1.05]),
        template="plotly_white",
        width=1100, height=500,
        legend=dict(orientation="h", y=1.12)
    )
    return fig


In [3]:
# 1. Carregamento dos Dados, Rótulos e Ortobase
PATH_GENES_FUJITA = DADOS_DIR / "fujita" / "featuresF.tsv"
PATH_GENES_PAN = DADOS_DIR / "pan" / "featuresPANcorrigido.tsv"
PATH_GENES_MATHYS = DADOS_DIR / "mathys" / "featuresM.tsv"

genes_fujita = carregar_ensembl_ids(PATH_GENES_FUJITA)
genes_pan = carregar_ensembl_ids(PATH_GENES_PAN)
genes_mathys = carregar_ensembl_ids(PATH_GENES_MATHYS)

s_triple = set(genes_pan) & set(genes_fujita) & set(genes_mathys)
mask_ruido = np.array([g not in s_triple for g in genes_pan])
print(f"Total genes Pan    : {len(genes_pan)}")
print(f"Genes na interseção: {len(s_triple)}")
print(f"Genes com ruído 0.5: {mask_ruido.sum()} ({mask_ruido.sum()/len(genes_pan)*100:.2f}%)")

orthbase_ref = np.load(DADOS_DIR / "pan" / "orthbase_pan_600d.npy").astype(np.float32)
X_limpo = np.load(DADOS_DIR / "pan" / "X_pan_600d.npy").astype(np.float32)
y_ref = np.load(DADOS_DIR / "pan" / "y_pan.npy")
X_ruido = np.load(DADOS_DIR / "pan" / "X_pan_ruido_600d.npy").astype(np.float32)

# TRAVAS DE SEGURANÇA (ASSERTIONS FAIL-FAST)
assert orthbase_ref.shape == (len(genes_pan), DIM_PROJECAO), f"Ortobase inválida: {orthbase_ref.shape}"
assert X_limpo.shape == (len(y_ref), DIM_PROJECAO), f"X_limpo inválido: {X_limpo.shape}"
assert X_ruido.shape == (len(y_ref), DIM_PROJECAO), f"X_ruido inválido: {X_ruido.shape}"
print("Todas as travas de integridade passaram com sucesso!")


Total genes Pan    : 61541
Genes na interseção: 30225
Genes com ruído 0.5: 31316 (50.89%)


Todas as travas de integridade passaram com sucesso!


In [4]:
# 2. Particionamento Estratificado (70% Treino / 15% Validação / 15% Teste Holdout)
indices = np.arange(len(y_ref))
idx_treino_val, idx_teste = train_test_split(
    indices, test_size=0.15, random_state=SEED, stratify=y_ref
)
razao_val = 0.15 / (1.0 - 0.15)
idx_treino, idx_val = train_test_split(
    idx_treino_val, test_size=razao_val, random_state=SEED, stratify=y_ref[idx_treino_val]
)

print(f"Treino    : {len(idx_treino)} células ({len(idx_treino)/len(indices)*100:.1f}%)")
print(f"Validação : {len(idx_val)} células ({len(idx_val)/len(indices)*100:.1f}%)")
print(f"Teste     : {len(idx_teste)} células ({len(idx_teste)/len(indices)*100:.1f}%)")

scaler = StandardScaler()
X_treino_s = scaler.fit_transform(X_limpo[idx_treino])
X_val_s = scaler.transform(X_limpo[idx_val])
X_teste_limpo_s = scaler.transform(X_limpo[idx_teste])

X_teste_ruido_s = scaler.transform(X_ruido[idx_teste])
X_full_ruido_s = scaler.transform(X_ruido)

joblib.dump(scaler, DADOS_DIR / "pan" / "scaler_exp04.pkl")


Treino    : 40219 células (70.0%)
Validação : 8619 células (15.0%)
Teste     : 8619 células (15.0%)


['/home/rasert/workspace/Teste_MlP_fujita_para_fujitafaltante_fujitapos_imputacao/dados/pan/scaler_exp04.pkl']

In [5]:
# 3. Treinamento do Classificador MLP
modelo = construir_modelo_mlp(input_dim=DIM_PROJECAO, n_classes=len(CLASSES_VALIDAS))
caminho_modelo = DADOS_DIR / "pan" / "modelo_mlp_exp04_pan_para_pan.keras"
callbacks = [
    EarlyStopping(monitor="val_loss", mode="min", patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint(str(caminho_modelo), monitor="val_loss", save_best_only=True, verbose=0)
]

t0 = time.time()
historico = modelo.fit(
    X_treino_s, y_ref[idx_treino],
    validation_data=(X_val_s, y_ref[idx_val]),
    epochs=50,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)
print(f"Treinamento concluído em {time.time() - t0:.2f} segundos.")


Epoch 1/50


W0000 00:00:1792859181.697405 1089807 gpu_device.cc:2342] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


  1/629 ━━━━━━━━━━━━━━━━━━━━ 9:06 870ms/step - accuracy: 0.2344 - loss: 1.8641

 25/629 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7000 - loss: 0.9695    

 42/629 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.7690 - loss: 0.7421

...

617/629 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step - accuracy: 0.9989 - loss: 0.0034

629/629 ━━━━━━━━━━━━━━━━━━━━ 2s 2ms/step - accuracy: 0.9989 - loss: 0.0034 - val_accuracy: 0.9920 - val_loss: 0.0534


Epoch 6: early stopping


Restoring model weights from the end of the best epoch: 1.


Treinamento concluído em 10.96 segundos.


In [6]:
# 4. Gráfico 1 — Curvas de Aprendizado (Acurácia e Loss)
fig_aprendizado = plotar_curvas_aprendizado(historico, "Experimento 04: Curvas de Aprendizado MLP (Pan Limpo)")
fig_aprendizado.show()


In [7]:
# 5. Avaliação Baseline Limpa (Conjunto de Teste Holdout 15%)
probs_teste_limpo = modelo.predict(X_teste_limpo_s, verbose=0)
y_pred_teste_limpo = np.argmax(probs_teste_limpo, axis=1)

relatorio_limpo = classification_report(
    y_ref[idx_teste], y_pred_teste_limpo,
    target_names=NOMES_CLASSES,
    output_dict=True
)
print("=== RELATÓRIO BASELINE: PAN TESTE 15% ===")
print(classification_report(y_ref[idx_teste], y_pred_teste_limpo, target_names=NOMES_CLASSES))

fig_cm_limpo = plotar_matriz_confusao(
    y_ref[idx_teste], y_pred_teste_limpo,
    "Experimento 04: Matriz de Confusão — Pan Teste Limpo (Baseline)"
)
fig_cm_limpo.show()


=== RELATÓRIO BASELINE: PAN TESTE 15% ===
              precision    recall  f1-score   support

    Classe 1       1.00      1.00      1.00      2830
    Classe 2       0.93      0.90      0.91        82
    Classe 3       0.97      0.98      0.98       887
    Classe 4       0.95      0.96      0.95       403
    Classe 5       1.00      0.96      0.98       260
    Classe 6       0.99      1.00      1.00      3658
    Classe 7       0.99      0.97      0.98       499

    accuracy                           0.99      8619
   macro avg       0.98      0.97      0.97      8619
weighted avg       0.99      0.99      0.99      8619



In [8]:
# 6. Avaliação com Ruído Sentinela 0,5 (Teste Holdout 15% e Dataset Completo 100%)
# 6.1 Teste Holdout com Ruído
probs_teste_ruido = modelo.predict(X_teste_ruido_s, verbose=0)
y_pred_teste_ruido = np.argmax(probs_teste_ruido, axis=1)

relatorio_ruido = classification_report(
    y_ref[idx_teste], y_pred_teste_ruido,
    target_names=NOMES_CLASSES,
    output_dict=True
)
print("=== RELATÓRIO: TESTE HOLDOUT COM RUÍDO 0,5 ===")
print(classification_report(y_ref[idx_teste], y_pred_teste_ruido, target_names=NOMES_CLASSES))

fig_cm_ruido = plotar_matriz_confusao(
    y_ref[idx_teste], y_pred_teste_ruido,
    "Experimento 04: Matriz de Confusão — Teste Holdout com Ruído 0,5"
)
fig_cm_ruido.show()

# 6.2 Dataset Completo 100% com Ruído
probs_full_ruido = modelo.predict(X_full_ruido_s, verbose=0)
y_pred_full_ruido = np.argmax(probs_full_ruido, axis=1)
print(f"Acurácia no Dataset Completo (100%) com Ruído: {accuracy_score(y_ref, y_pred_full_ruido):.4f}")


=== RELATÓRIO: TESTE HOLDOUT COM RUÍDO 0,5 ===
              precision    recall  f1-score   support

    Classe 1       0.99      0.98      0.99      2830
    Classe 2       0.87      0.80      0.84        82
    Classe 3       0.99      0.87      0.93       887
    Classe 4       0.89      0.93      0.91       403
    Classe 5       1.00      0.42      0.59       260
    Classe 6       0.91      1.00      0.95      3658
    Classe 7       1.00      0.79      0.88       499

    accuracy                           0.95      8619
   macro avg       0.95      0.83      0.87      8619
weighted avg       0.95      0.95      0.94      8619



Acurácia no Dataset Completo (100%) com Ruído: 0.9467


In [9]:
# 7. Gráfico 4 — Comparativo de Desempenho (Baseline Limpo vs. Ruído 0,5)
fig_comp = plotar_comparativo(
    relatorio_limpo, relatorio_ruido,
    "Experimento 04 (Pan -> Pan): Queda de Desempenho com Ruído 0,5",
    nome_cenario_ruido="Pan com Ruído 0,5"
)
fig_comp.show()

acc_limpo = relatorio_limpo["accuracy"]
acc_ruido = relatorio_ruido["accuracy"]
macro_limpo = relatorio_limpo["macro avg"]["f1-score"]
macro_ruido = relatorio_ruido["macro avg"]["f1-score"]

print("=" * 60)
print(f"Acurácia Global   : {acc_limpo*100:.2f}% -> {acc_ruido*100:.2f}% (Queda de {(acc_limpo - acc_ruido)*100:+.2f} p.p.)")
print(f"Macro F1-Score    : {macro_limpo*100:.2f}% -> {macro_ruido*100:.2f}% (Queda de {(macro_limpo - macro_ruido)*100:+.2f} p.p.)")
print("=" * 60)


Acurácia Global   : 99.04% -> 94.65% (Queda de +4.39 p.p.)
Macro F1-Score    : 97.13% -> 86.93% (Queda de +10.20 p.p.)


# 8. Avaliação Pós-Imputação Hopfield (Reconstrução de Features Ausentes)

## Objetivo
Avaliar a capacidade de classificação do modelo **MLP (treinado no baseline limpo)** após a substituição das features faltantes (anteriormente preenchidas com ruído `0,5`) pelos valores transcricionais binários inferidos pela **Modern Hopfield Network** e reprojetados no espaço **SWeeP 600D**.

- **Cenário:** Experimento 04: Pan para Pan (Intra-Dataset)
- **Projeção Pós-Hopfield:** `dados/pan/X_pan_pos_hopfield_600d.npy`
- **Avaliação:** Comparação pareada contra o *Baseline Limpo* e o cenário *Com Ruído 0,5*.


In [ ]:
# 8.1 Inferência Pós-Hopfield no Conjunto de Teste Holdout (15% Cego) e Dataset Completo
import os
import joblib
import numpy as np
from pathlib import Path
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
import tensorflow as tf
from tensorflow.keras.models import load_model

tf.config.set_visible_devices([], "GPU")

if Path.cwd().name == "experimentos":
    os.chdir(Path.cwd().parent)
ROOT_DIR = Path.cwd()

DIM_PROJECAO = 600
CLASSES_VALIDAS = [1, 2, 3, 4, 5, 6, 7]
NOMES_CLASSES = [f"Classe {i}" for i in CLASSES_VALIDAS]

modelo = load_model(ROOT_DIR / "dados/pan/modelo_mlp_exp04_pan_para_pan.keras")
scaler = joblib.load(ROOT_DIR / "dados/pan/scaler_exp04.pkl")

y_ref = np.load(ROOT_DIR / "dados/pan/y_pan.npy")
indices = np.arange(len(y_ref))
idx_treino_val, idx_teste = train_test_split(indices, test_size=0.15, random_state=42, stratify=y_ref)

X_pos_hopfield = np.load(ROOT_DIR / "dados/pan/X_pan_pos_hopfield_600d.npy").astype(np.float32)
assert X_pos_hopfield.shape == (len(y_ref), DIM_PROJECAO), f"Shape inválido: {X_pos_hopfield.shape}"

X_teste_hop_s = scaler.transform(X_pos_hopfield[idx_teste])
probs_teste_hop = modelo.predict(X_teste_hop_s, verbose=0)
y_pred_teste_hop = np.argmax(probs_teste_hop, axis=1)

print("=== RELATÓRIO: TESTE HOLDOUT PÓS-IMPUTAÇÃO HOPFIELD ===")
print(classification_report(y_ref[idx_teste], y_pred_teste_hop, target_names=NOMES_CLASSES))

X_full_hop_s = scaler.transform(X_pos_hopfield)
probs_full_hop = modelo.predict(X_full_hop_s, verbose=0)
y_pred_full_hop = np.argmax(probs_full_hop, axis=1)
acc_full_hop = accuracy_score(y_ref, y_pred_full_hop)
print(f"Acurácia no Dataset Completo (100%) Pós-Hopfield: {acc_full_hop:.4f}")


=== RELATÓRIO: TESTE HOLDOUT PÓS-IMPUTAÇÃO HOPFIELD ===
              precision    recall  f1-score   support

    Classe 1       1.00      1.00      1.00      2830
    Classe 2       0.93      0.90      0.91        82
    Classe 3       0.98      0.98      0.98       887
    Classe 4       0.94      0.97      0.95       403
    Classe 5       1.00      0.96      0.98       260
    Classe 6       1.00      1.00      1.00      3658
    Classe 7       0.99      0.97      0.98       499

    accuracy                           0.99      8619
   macro avg       0.98      0.97      0.97      8619
weighted avg       0.99      0.99      0.99      8619

Acurácia no Dataset Completo (100%) Pós-Hopfield: 0.9955


In [ ]:
# 8.2 Matriz de Confusão Pós-Imputação Hopfield
import plotly.graph_objects as go
from sklearn.metrics import confusion_matrix

def plotar_matriz_confusao(y_true, y_pred, titulo):
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(NOMES_CLASSES)))
    cm_norm = cm.astype(float) / np.maximum(cm.sum(axis=1, keepdims=True), 1e-9)
    fig = go.Figure(data=go.Heatmap(
        z=cm_norm, x=NOMES_CLASSES, y=NOMES_CLASSES, colorscale="Blues",
        text=[[f"{cm[i, j]}<br>({cm_norm[i, j]*100:.1f}%)" for j in range(len(NOMES_CLASSES))] for i in range(len(NOMES_CLASSES))],
        texttemplate="%{text}", textfont={"size": 11}
    ))
    fig.update_layout(title=titulo, xaxis_title="Classe Prevista", yaxis_title="Classe Verdadeira", template="plotly_white", width=750, height=650)
    return fig

fig_cm_hop = plotar_matriz_confusao(y_ref[idx_teste], y_pred_teste_hop, "Experimento 04: Pan para Pan (Intra-Dataset): Matriz de Confusão — Teste Holdout Pós-Hopfield")
fig_cm_hop.show()


<Figure size 750x650 at 0x759181c732f0>

In [ ]:
# 8.3 Comparativo de Desempenho Triplo (Limpo vs. Ruído 0,5 vs. Pós-Hopfield)
import json
import plotly.graph_objects as go

with open(ROOT_DIR / "dados/pan/relatorio_recuperacao_exp04.json", "r") as fj:
    dados_resumo = json.load(fj)

rotulos_plot = ["Acurácia Global", "Macro F1", "Weighted F1"] + [f"F1 {nome}" for nome in NOMES_CLASSES]
limpo = dados_resumo["baseline_limpo"]
ruido = dados_resumo["com_ruido_05"]
hop = dados_resumo["pos_hopfield"]

vals_limpo = [limpo["acuracia"], limpo["macro_f1"], limpo["weighted_f1"]] + [limpo["f1_classes"].get(n, 0.0) for n in NOMES_CLASSES]
vals_ruido = [ruido["acuracia"], ruido["macro_f1"], ruido["weighted_f1"]] + [ruido["f1_classes"].get(n, 0.0) for n in NOMES_CLASSES]
vals_hop = [hop["acuracia"], hop["macro_f1"], hop["weighted_f1"]] + [hop["f1_classes"].get(n, 0.0) for n in NOMES_CLASSES]

fig_triplo = go.Figure(data=[
    go.Bar(name="Baseline (Sem Ruído)", x=rotulos_plot, y=vals_limpo, marker_color="#2ca02c"),
    go.Bar(name="Com Ruído 0,5", x=rotulos_plot, y=vals_ruido, marker_color="#d62728"),
    go.Bar(name="Pós-Imputação Hopfield", x=rotulos_plot, y=vals_hop, marker_color="#1f77b4")
])
fig_triplo.update_layout(title="Experimento 04: Pan para Pan (Intra-Dataset): Comparativo de Recuperação Pós-Hopfield", barmode="group", yaxis=dict(title="Pontuação (0.0 a 1.0)", range=[0, 1.05]), template="plotly_white", width=1150, height=520)
fig_triplo.show()

rec = dados_resumo["recuperacao"]
print("=" * 70)
print("  RESUMO DE RECUPERAÇÃO TRANSCRICIONAL VIA MODERN HOPFIELD NETWORK")
print("=" * 70)
print(f"Acurácia Global  : Limpo {limpo['acuracia']*100:.2f}% | Ruído {ruido['acuracia']*100:.2f}% | Hopfield {hop['acuracia']*100:.2f}% (Delta: {rec['delta_acuracia_pp']:+.2f} p.p.)")
print(f"Macro F1-Score   : Limpo {limpo['macro_f1']*100:.2f}% | Ruído {ruido['macro_f1']*100:.2f}% | Hopfield {hop['macro_f1']*100:.2f}% (Delta: {rec['delta_macro_f1_pp']:+.2f} p.p.)")
print(f"Taxa de Recuperação da Queda de Macro F1: {rec['taxa_recuperacao_macro_pct']:.2f}%")
print("=" * 70)


  RESUMO DE RECUPERAÇÃO TRANSCRICIONAL VIA MODERN HOPFIELD NETWORK
Acurácia Global  : Limpo 99.04% | Ruído 94.65% | Hopfield 99.07% (Delta: +4.42 p.p.)
Macro F1-Score   : Limpo 97.13% | Ruído 86.93% | Hopfield 97.14% (Delta: +10.22 p.p.)
Taxa de Recuperação da Queda de Macro F1: 100.15%
Relatório comparativo salvo com sucesso em: /home/rasert/workspace/wt_hopfield_pan/dados/pan/relatorio_recuperacao_exp04.json
